## Modeling gold — the patterns BI and ML want

Two patterns cover most gold tables. Choose by **consumer**.

**Star schema** — a central fact surrounded by conformed dimensions:

- `gold.fact_card_transactions` — one row per transaction, foreign keys to dimensions.
- `gold.dim_customer`, `gold.dim_merchant`, `gold.dim_date`.

Dimensions usually slowly-change. When a customer's city changes, **SCD type 2** captures the change while preserving history — `APPLY CHANGES INTO ... STORED AS SCD TYPE 2` is the right tool. Star schema is what **Power BI / Tableau** dashboards want.

**Wide rollup / OBT (one big table)** — one row per business entity, every interesting metric inline:

- `gold.customer_360` — `customer_id` + 30+ rollup columns. Easier for analysts (no joins) but more expensive to refresh.

OBT is what **ad-hoc analytics and ML feature engineering** want.

**The bank's gold layout that ties the module together:**

```text
gold.customer_360            — materialized view, refresh hourly
gold.daily_card_volume       — streaming table, continuous
gold.dim_customer            — APPLY CHANGES INTO ... SCD TYPE 2
gold.fact_card_transactions  — APPLY CHANGES INTO ... SCD TYPE 1
```

**The exam's angle:** match the *consumer* to the *shape* — BI dashboards → star schema; ML features / ad-hoc → one big table — and match the *object* to the *refresh pattern* (MV for scheduled aggregation, streaming table for continuous, `APPLY CHANGES` for CDC dimensions).
